# 10d Evaluate and Refine Concepts

Build yearly abstract-prevalence signals and score them with fixed direct forecasts. Run this notebook during discovery rounds; edit `CURRENT_POPULATION`, `CANDIDATE_IDS`, and `PARENT_CHILD_REPLACEMENTS` after reviewing the development results. Confirmation pairs are not used to choose concepts.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from amr.pipeline_config import cfg, load_pipeline_config
from amr.pipelines.concept_notebook_workflows import evaluate_development_concepts
from amr.pipeline_config import cfg, load_pipeline_config
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
FIT_START_YEAR = int(cfg(PIPELINE_CONFIG, "si.fit_start_year", 2000))
FORECAST_ORIGIN_END_YEAR = int(cfg(PIPELINE_CONFIG, "si.fit_end_year", 2023))
LAG_LENGTH = 3
FORECAST_HORIZONS = (1, 2, 3)
MIN_PRIOR_OBSERVATIONS = 10
FORECAST_PROGRESS_EVERY = 200
FORECAST_MAX_WORKERS = 4  # Independent model/horizon fits; set -1 to use all available CPUs.


## Configure the discovery round

`CURRENT_POPULATION` lists concepts retained from the previous development round. `CANDIDATE_IDS` lists concepts to test. Each candidate is compared with the SI-only baseline and, when a population is specified, with that population. These conditional gains measure whether each selected concept contributes beyond the other population members. `PARENT_CHILD_REPLACEMENTS` tests whether child signals predict better than their parent.

Start with an empty population to inspect seed concepts individually. After each round, use development results and scientific interpretability to decide what to retain, add, or refine. Do not use notebook 10e confirmation results for these choices.

In [ ]:
CONTEXT = load_concept_catalog_context(PROJECT_ROOT)
CONCEPT_IDS = [str(concept["concept_id"]) for concept in CONTEXT["catalog"]]
CURRENT_POPULATION = []  # Concepts retained from the previous development round.
CANDIDATE_IDS = CONCEPT_IDS
PARENT_CHILD_REPLACEMENTS = {}  # e.g. {"surveillance": ["genomic_surveillance", "hospital_surveillance"]}


## Development rolling-origin forecasts

At every eligible origin, the model uses `LAG_LENGTH` prior annual SI values and the same number of prior annual values for each selected concept to forecast directly at each horizon in `FORECAST_HORIZONS`. It fits pair-specific intercepts and shared lag slopes. A pair needs at least `MIN_PRIOR_OBSERVATIONS` prior observed SI values; missing SI is not smoothed or interpolated.

Forecast rows are saved so later review does not require recomputing Jev labels. The forecast runs independent model/horizon fits concurrently using `FORECAST_MAX_WORKERS`, and prints progress every `FORECAST_PROGRESS_EVERY` model-origin steps with elapsed time and ETA. A lower error means a concept adds predictive information under this setup; it does not show that the concept causes interest to change.

In [ ]:
RESULT = evaluate_development_concepts(
    PROJECT_ROOT, current_population=CURRENT_POPULATION, candidate_ids=CANDIDATE_IDS,
    parent_child_replacements=PARENT_CHILD_REPLACEMENTS, fit_start_year=FIT_START_YEAR,
    fit_end_year=FORECAST_ORIGIN_END_YEAR, lag_length=LAG_LENGTH, horizons=FORECAST_HORIZONS,
    min_prior_observations=MIN_PRIOR_OBSERVATIONS, progress_every=FORECAST_PROGRESS_EVERY,
    max_workers=FORECAST_MAX_WORKERS,
)
display(RESULT["population_summary"].sort_values("generality", ascending=False))


## Horizon-specific and concept-level results

A positive `forecast_gain` means lower mean squared error than the reference model. `generality` averages gain across eligible pairs; `specialist_strength` is the 90th percentile of pair-level gain. Read each horizon alongside the equal-weight aggregate: a concept may help at one horizon and hurt at another. Check pair counts before interpreting a strong specialist score as broad evidence.

In [ ]:
display(RESULT["pair_summary"].groupby(["model", "horizon"], as_index=False).agg(
    generality=("forecast_gain", "mean"),
    specialist_strength=("forecast_gain", lambda values: values.quantile(0.9)),
    pairs=("forecast_gain", "count"), forecasts=("forecasts", "sum"),
).sort_values(["model", "horizon"]))
